In [ ]:
# src/pipeline.py

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor


class AdvancedDateEncoder(BaseEstimator, TransformerMixin):

    def __init__(self, date_column="date"):
        self.date_column = date_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.date_column] = pd.to_datetime(X[self.date_column])

        X["month"] = X[self.date_column].dt.month
        X["dayofyear"] = X[self.date_column].dt.dayofyear
        X["quarter"] = X[self.date_column].dt.quarter
        X["dayofweek"] = X[self.date_column].dt.dayofweek

        X["month_sin"] = np.sin(2 * np.pi * X["month"] / 12)
        X["month_cos"] = np.cos(2 * np.pi * X["month"] / 12)

        X["dayofyear_sin"] = np.sin(2 * np.pi * X["dayofyear"] / 365)
        X["dayofyear_cos"] = np.cos(2 * np.pi * X["dayofyear"] / 365)

        X["is_weekend"] = (X["dayofweek"] >= 5).astype(int)
        X["is_monsoon"] = X["month"].isin([6, 7, 8, 9]).astype(int)

        return X.drop(columns=[self.date_column, "month", "dayofyear", "dayofweek"])


def build_pipeline():

    numeric_pipeline = Pipeline([
        ("imputer", IterativeImputer()),
        ("scaler", RobustScaler())
    ])

    categorical_pipeline = Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    model_pipeline = Pipeline([
        ("date_features", AdvancedDateEncoder(date_column="date")),
        ("preprocessing", ColumnTransformer([
            ("num", numeric_pipeline,
             make_column_selector(dtype_include=np.number)),
            ("cat", categorical_pipeline,
             make_column_selector(dtype_exclude=np.number))
        ], remainder="drop")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_split=5,
            random_state=42
        ))
    ])

    return model_pipeline